# 2026-03-05 TODO

|TODO|내용|비고|
|:---:|:---:|:---:|
|[v]|1. GCP BigQuery에서 Polars으로 상담 코드 분류||
|[v]|2. GCP BigQuery에서 Polars으로 상담 멘트 행분리 정렬화 | 멘트가 종료시 행 생성, 문장 종료 횟수는 `end_content_index`이라는 행에다가 카운트 할것.|

In [1]:
!pip install 'google-cloud-bigquery[bqstorage,pandas]' polars pyarrow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
from google.cloud import bigquery
from google.oauth2 import service_account

key_path = '/workspaces/Psychological-counseling-researching/.key/testprojects-453622-d1f78fcce8b7.json'
credentials = service_account.Credentials.from_service_account_file(key_path)

client = bigquery.Client(credentials=credentials, project=credentials.project_id)
print(f"Client created with project: {client.project}")


Client created with project: testprojects-453622


## 1. GCP BigQuery에서 Polars으로 상담 코드 분류

In [3]:
import polars as pl
from google.cloud import bigquery

# Perform a query.
QUERY = open('/workspaces/Psychological-counseling-researching/1-text_data/2_test.sql', 'r', encoding='utf-8').read()

query_job = client.query(QUERY)  # API request
rows = query_job.result()  # Waits for query to finish

raw_df = pl.from_arrow(rows.to_arrow())
display(raw_df)

file_code,max_session_no
str,i64
"""A024""",2
"""A099""",2
"""A094""",2
"""A100""",3
"""A129""",4
…,…
"""A081""",10
"""A115""",11
"""A086""",11


## 2. GCP BigQuery에서 Polars으로 상담 멘트 행분리 정렬화

|TODO|내용|구현일시|
|:---:|:---:|:---:|
|[v]|텍스트 토큰나이저를 기반으로 진행 해야할것. 모델은 대충 [Kiwi](https://github.com/bab2min/kiwipiepy)를 가지고 구축 할것.|2026/03/05 - 2026/03/06|


In [4]:
!pip install kiwipiepy


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [5]:
TEST_QUERY = """
SELECT *
FROM
    `Psychological_counseling_data.processed_data`
ORDER BY
    file_code ASC,
    index ASC;
"""

In [8]:
query_job = client.query(TEST_QUERY)  # API request
rows = query_job.result()  # Waits for query to finish

raw_df = pl.from_arrow(rows.to_arrow())
display(raw_df[0:2])

index,FilePath,Speaker,Content,FileName,prefix,topic,session_no,stage,file_code,file_code_row_count,speaker_row_count,file_code_ratio_pct,speaker_ratio_pct
i64,str,str,str,str,str,str,i64,str,str,i64,i64,f64,f64
0,"""gs://testprojects-453622-unzip…","""상담사""","""오늘 선생님과 세 번째 상담 시작하도록 할게요. 알겠습…","""resource_addiction_3_check_A00…","""resource""","""addiction""",3,"""check""","""A008""",2051,39338,2.6126,50.1089
0,"""gs://testprojects-453622-unzip…","""상담사""","""선생님 반갑습니다. 저희 이제 선생님하고 8회기 동안 …","""resource_addiction_1_check_A00…","""resource""","""addiction""",1,"""check""","""A008""",2051,39338,2.6126,50.1089


In [22]:
from kiwipiepy import Kiwi
import polars as pl

kiwi = Kiwi()

# 문장 종료 규칙: 종결 어미(EF) 뒤에 종결 부호/줄임표/붙임표(SF/SE/SO)가 나올 때만 종료
END_EOMI_TAG = "EF"
END_SYMBOL_TAGS = {"SF", "SE", "SO"}


def split_sentences_by_pos(text: str) -> list[str]:
    if text is None:
        return []

    tokens = kiwi.tokenize(str(text))
    results: list[str] = []
    buffer: list[str] = []
    waiting_end_symbol = False

    for tok in tokens:
        buffer.append(tok.form)

        if tok.tag == END_EOMI_TAG:
            waiting_end_symbol = True
            continue

        if waiting_end_symbol and tok.tag in END_SYMBOL_TAGS:
            sent = "".join(buffer).strip()
            if sent:
                results.append(sent)
            buffer = []
            waiting_end_symbol = False
            continue

        waiting_end_symbol = False

    return results


def resolve_col(cols: list[str], candidates: list[str], required: bool = True) -> str | None:
    col_map = {c.lower(): c for c in cols}
    for cand in candidates:
        found = col_map.get(cand.lower())
        if found is not None:
            return found
    if required:
        raise ValueError(f"필수 컬럼이 없습니다. 후보={candidates}, 현재={cols}")
    return None


file_code_col = resolve_col(raw_df.columns, ["file_code"])
content_col = resolve_col(raw_df.columns, ["content"])
speaker_col = resolve_col(raw_df.columns, ["speaker"], required=False)
session_no_col = resolve_col(raw_df.columns, ["session_no"], required=False)
index_col = resolve_col(raw_df.columns, ["index"], required=False)

sort_cols = [file_code_col]
if index_col is not None:
    sort_cols.append(index_col)

base_df = (
    raw_df.with_row_index("_orig_order")
    .sort(sort_cols + ["_orig_order"])
    .with_columns((pl.col(file_code_col).cum_count().over(file_code_col) - 1).alias("timeline_index"))
)

records = []
for row in base_df.iter_rows(named=True):
    split_list = split_sentences_by_pos(row.get(content_col))

    for end_content_index, sent in enumerate(split_list):
        records.append(
            {
                "file_code": row[file_code_col],
                "speaker": row.get(speaker_col) if speaker_col is not None else None,
                "session_no": row.get(session_no_col) if session_no_col is not None else None,
                "timeline_index": row["timeline_index"],
                "end_content_index": end_content_index,
                "split_contents": sent,
            }
        )

if records:
    split_df = (
        pl.DataFrame(records)
        .sort(["file_code", "timeline_index", "end_content_index"])
        .with_columns((pl.col("file_code").cum_count().over("file_code") - 1).alias("split_row_index"))
        .select(
            [
                "file_code",
                "speaker",
                "session_no",
                "timeline_index",
                "end_content_index",
                "split_row_index",
                "split_contents",
            ]
        )
    )
else:
    split_df = pl.DataFrame(
        schema={
            "file_code": pl.Utf8,
            "speaker": pl.Utf8,
            "session_no": pl.Utf8,
            "timeline_index": pl.Int64,
            "end_content_index": pl.Int64,
            "split_row_index": pl.Int64,
            "split_contents": pl.Utf8,
        }
    )

display(split_df)

file_code,speaker,session_no,timeline_index,end_content_index,split_row_index,split_contents
str,str,i64,i64,i64,u32,str
"""A008""","""상담사""",3,0,0,0,"""오늘선생님과세번째상담시작하도록할게요."""
"""A008""","""상담사""",3,0,1,1,"""알겠습니다."""
"""A008""","""상담사""",1,1,0,2,"""선생님반갑습니다."""
"""A008""","""상담사""",1,1,1,3,"""저희이제선생님하고8회기동안상담을진행하게될거이고요.저는…"
"""A008""","""상담사""",6,2,0,4,"""6회기상담시작하도록하겠습니다."""
…,…,…,…,…,…,…
"""A139""","""상담사""",3,1455,1,2909,"""이것도한번점검한것도같이얘기하어보고싶은것이많네요."""
"""A139""","""상담사""",3,1455,2,2910,"""다음회기때한번얘기를하어보면좋을것같어요."""
"""A139""","""상담사""",3,1457,0,2911,"""네,한주도잘지내시고월요일에뵙겠습니다."""


In [23]:
# 데이터프레임을 Pandas로 변환
pandas_df = split_df.to_pandas()

# BigQuery에 업로드
project_id = credentials.project_id
table_id = f'{project_id}.Psychological_counseling_data.morpheme_classification'

pandas_df.to_gbq(
    destination_table=table_id,
    project_id=project_id,
    if_exists='replace',  # 테이블이 이미 존재하면 덮어쓰기
    credentials=credentials
)

print(f"Table {table_id} uploaded successfully.")

/tmp/ipykernel_48134/839509728.py:8: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  pandas_df.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 12018.06it/s]

Table testprojects-453622.Psychological_counseling_data.morpheme_classification uploaded successfully.
